# Filename classification experiments

This notebook runs the shared comparison for TF-IDF + Multinomial Naive Bayes, TF-IDF + Logistic Regression, and TF-IDF + TruncatedSVD + Logistic Regression. It uses 1,200 **synthetic** filenames (200 per category); results are development comparisons and do not estimate real-world performance. The highest-scoring model is selected on the comparison holdout, so the reported scores are not an independent final estimate.

All filename variants from one stem are grouped into the same split. This prevents near-duplicate suffix versions from leaking between training and test data. SVD is paired with Logistic Regression because SVD produces latent values that can be negative, while Multinomial Naive Bayes requires nonnegative features.

Run this notebook from the repository root.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.ml_pipeline import load_dataset, train_and_evaluate

ROOT = Path.cwd()
if not (ROOT / "data" / "file_dataset.csv").is_file():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

data = load_dataset()
print(f"Rows: {len(data)}")
print(data.head().to_string(index=False))
print(data["category"].value_counts().sort_index().rename("count").to_string())

## Train and compare on the shared split

The experiment function uses a fixed random seed and a category-stratified 80/20 split of filename families. It records class metrics, confusion matrices, misclassifications, runtime, and SVD component count, then persists the selected complete pipeline and report artifacts.

In [ ]:
metrics = train_and_evaluate()
print("Selected model:", metrics["selected_model"])
summary = pd.DataFrame(
    {
        name: {
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "weighted_f1": result["weighted_f1"],
            "runtime_seconds": result["runtime_seconds"],
            "misclassification_count": len(result["misclassifications"]),
            "svd_components": result["svd_components"],
        }
        for name, result in metrics["models"].items()
    }
).T
print(summary.to_string())

In [ ]:
for name, result in metrics["models"].items():
    print(f"\n{name}: per-class metrics")
    print(pd.DataFrame(result["per_class"]).T.to_string())
    print("Confusion matrix (rows=true, columns=predicted):")
    print(pd.DataFrame(
        result["confusion_matrix"],
        index=metrics["evaluation"]["labels"],
        columns=metrics["evaluation"]["labels"],
    ).to_string())
    print("Misclassifications:")
    print(pd.DataFrame(result["misclassifications"]).to_string(index=False))

In [ ]:
print("Saved artifacts:")
for artifact in (
    ROOT / "models" / "best_model.joblib",
    ROOT / "reports" / "model_metrics.json",
    ROOT / "reports" / "confusion_matrix.png",
):
    print(artifact.relative_to(ROOT), artifact.stat().st_size, "bytes")

print("Dataset provenance:", metrics["dataset"]["provenance"])
print("Split:", json.dumps(metrics["evaluation"], indent=2))